# Experiment 3: Advanced Prompt Engineering - Zero-shot, Few-shot and Chain-of-Thought

**Course:** HAAI302COM - Artificial Intelligence Laboratory

## 1. Aim

To design and evaluate zero-shot, few-shot and Chain-of-Thought (CoT) prompting strategies for different NLP tasks using an LLM API, and to quantify the accuracy improvement obtained from each prompting technique.

## 2. Objectives

- To call a large language model through an API from Python.
- To design zero-shot and few-shot prompts for a text classification task.
- To design direct, zero-shot CoT and few-shot CoT prompts for a maths word-problem task.
- To measure accuracy, token usage and response time for every strategy.
- To analyse where each prompting technique helps and where it does not.

## 3. Software Requirements

- Python 3.9 or later
- Google Colab or Jupyter Notebook
- An API key for the language model. This experiment uses the OpenAI Python SDK pointed at the Gemini OpenAI-compatible endpoint (model `gemini-2.5-flash-lite`). The same code works with the OpenAI or Anthropic APIs by changing the client and the model name.
- Required Python libraries: `openai`, `python-dotenv`, `datasets`, `pandas`

## 4. Theory

### A. Zero-shot prompting

The model receives only an instruction and the input. It must rely on what it learned during pre-training.

`Classify the emotion of the text. Text: I am so happy today. Label:`

### B. Few-shot prompting

The prompt contains a few worked examples (input and correct output) before the real input. The model infers the task, the output format and the label vocabulary from the examples (in-context learning). No weights are updated.

### C. Chain-of-Thought (CoT) prompting

For problems that need several steps, the model is asked to write its reasoning before the final answer. This gives it intermediate tokens to compute with.

- **Zero-shot CoT:** add an instruction such as "Let's think step by step."
- **Few-shot CoT:** show worked examples whose answers contain the reasoning steps.

### D. Evaluation

Accuracy is the fraction of test questions answered correctly. The answer must be extracted from the reply and compared with the known correct answer. Cost is measured in tokens, and speed in seconds per request.

| Task | Dataset | Strategies compared |
|---|---|---|
| Emotion classification (6 classes) | `dair-ai/emotion` | zero-shot, few-shot with 6 examples, few-shot with 18 examples |
| Grade-school maths word problems | `openai/gsm8k` | direct answer, zero-shot CoT, few-shot CoT |

## 5. Algorithm / Procedure

1. Install and import the libraries and create the API client.
2. Write a helper function that sends a prompt and returns the reply with its token count and time.
3. Load the emotion dataset and build zero-shot and few-shot prompts.
4. Run every prompt on the test set and compute the accuracy.
5. Load the GSM8K dataset and build direct, zero-shot CoT and few-shot CoT prompts.
6. Extract the numeric answer from every reply and compute the accuracy.
7. Compare accuracy, tokens and time in a table and record observations.

## 6. Python Program

### Step 1: Install the libraries and create the client

In [1]:
!pip install -q openai python-dotenv datasets pandas

In [2]:
import os, re, time, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))   # reads GEMINI_API_KEY from the .env file
client = OpenAI(api_key=os.environ["GEMINI_API_KEY"],
                base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
MODEL = "gemini-2.5-flash-lite"

### Step 2: A helper to call the model

`ask()` sends one prompt with temperature 0 (the most repeatable setting) and retries if the service is busy. It returns the reply, the number of tokens generated and the time taken. `run_all()` sends many prompts in parallel to save time.

In [3]:
def ask(prompt):
    for attempt in range(5):
        try:
            t0 = time.time()
            r = client.chat.completions.create(model=MODEL, temperature=0, messages=[{"role": "user", "content": prompt}])
            return {"reply": r.choices[0].message.content or "", "tokens": r.usage.total_tokens, "seconds": time.time() - t0}
        except Exception:
            time.sleep(2 ** attempt)
    return {"reply": "", "tokens": 0, "seconds": 0}

def run_all(prompts):
    with ThreadPoolExecutor(8) as pool:
        return list(pool.map(ask, prompts))

print(ask("What is few-shot prompting? Answer in one sentence.")["reply"])

Few-shot prompting is a technique where a language model is provided with a small number of examples (typically 1-5) of a task before being asked to perform that task on new, unseen data.


### Step 3: Task A - emotion classification (zero-shot and few-shot)

The dataset has short English sentences labelled with one of six emotions. We test on 60 random sentences. The few-shot prompts add 6 (one per class) or 18 (three per class) labelled examples taken from the training set.

In [4]:
from datasets import load_dataset

train = load_dataset("dair-ai/emotion", split="train")
test = load_dataset("dair-ai/emotion", split="test").shuffle(seed=1).select(range(60))
LABELS = train.features["label"].names
print(LABELS)
pd.DataFrame({"text": test["text"][:4], "label": [LABELS[i] for i in test["label"][:4]]})

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


,text,label
0,i had the feeling that it might not have been ...,joy
1,im feeling pretty disheartened by the whole thing,sadness
2,im still feeling a little groggy from the lack...,sadness
3,i dont want to always be judgmental of particu...,sadness


In [5]:
INSTRUCTION = f"Classify the emotion of the text into exactly one of: {', '.join(LABELS)}. Reply with only the label."
pool = train.shuffle(seed=3).select(range(600))

def shots(per_class):
    return [e for c in range(6) for e in [x for x in pool if x["label"] == c][:per_class]]

def make_prompt(text, per_class=0):
    demos = "".join(f"Text: {e['text']}\nLabel: {LABELS[e['label']]}\n\n" for e in shots(per_class))
    return f"{INSTRUCTION}\n\n{demos}Text: {text}\nLabel:"

print(make_prompt(test[0]["text"], per_class=1))

Classify the emotion of the text into exactly one of: sadness, joy, love, anger, fear, surprise. Reply with only the label.

Text: i normally feel kind of awkward at birthday dinner parties since theres always someone i dont know but not this time
Label: sadness

Text: i feel that even though some bloggers are popular within one clique there are twice the amount of people who are jealous of their success but chooses to kiss their butt to fit in and triple who disrespect them for their pompous notoriety
Label: joy

Text: i feel specially fond of
Label: love

Text: i start to feel agitated inside
Label: anger

Text: im sore and feeling very unsure of how in the world i will go more miles in weeks
Label: fear

Text: i am sure at least i hope so that the woman who responded by saying so that he could help out with the kids also feel this way but what surprised me was that all the reasons i listed above were second
Label: surprise

Text: i had the feeling that it might not have been taken a

In [6]:
def parse_label(reply):
    word = re.sub(r"[^a-z]", "", reply.strip().lower().split("\n")[0])
    return word if word in LABELS else "invalid"

gold = [LABELS[i] for i in test["label"]]
rows = []
for name, k in [("zero-shot", 0), ("few-shot (6 examples)", 1), ("few-shot (18 examples)", 3)]:
    results = run_all([make_prompt(t, k) for t in test["text"]])
    preds = [parse_label(r["reply"]) for r in results]
    rows.append({"Strategy": name, "Accuracy (%)": round(100 * sum(p == g for p, g in zip(preds, gold)) / 60, 1),
                 "Avg tokens": round(sum(r["tokens"] for r in results) / 60)})
emotion_table = pd.DataFrame(rows).set_index("Strategy")
emotion_table

,Accuracy (%),Avg tokens
Strategy,,
zero-shot,61.7,57
few-shot (6 examples),66.7,235
few-shot (18 examples),66.7,595


### Step 4: Task B - maths word problems (direct, zero-shot CoT, few-shot CoT)

GSM8K contains multi-step arithmetic problems with a known numeric answer. We test on 30 problems. The three prompts differ only in how much reasoning the model is asked to show.

In [7]:
gsm_train = load_dataset("openai/gsm8k", "main", split="train")
gsm_test = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=7).select(range(30))
gold_num = [float(e["answer"].split("####")[-1].replace(",", "")) for e in gsm_test]
print(gsm_test[0]["question"], "\nCorrect answer:", gold_num[0])

Ashley's pizza delivery costs $15. What is the total amount that Ashley should give the delivery man if she wants to give a tip that is equal to 1/5 of the amount she ordered? 
Correct answer: 18.0


In [8]:
def worked(e):   # turn a GSM8K training example into a worked example with a final 'Answer:' line
    steps = re.sub(r"<<[^>]*>>", "", e["answer"].split("####")[0]).strip()
    return f"Q: {e['question']}\nA: {steps}\nAnswer: {e['answer'].split('####')[-1].strip()}"

examples = "\n\n".join(worked(e) for e in gsm_train.select(range(3)))
COT = "\nLet's think step by step. End with a final line 'Answer: <number>'."
prompts = {
    "direct answer":   lambda q: q + "\nRespond with only the final number, no words or working.",
    "zero-shot CoT":   lambda q: q + COT,
    "few-shot CoT":    lambda q: f"{examples}\n\nQ: {q}\nA:",
}
print(prompts["zero-shot CoT"](gsm_test[0]["question"]))

Ashley's pizza delivery costs $15. What is the total amount that Ashley should give the delivery man if she wants to give a tip that is equal to 1/5 of the amount she ordered?
Let's think step by step. End with a final line 'Answer: <number>'.


In [9]:
def extract_number(reply):
    text = reply.replace(",", "").replace("$", "")
    text = text.split("Answer:")[-1]
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return float(nums[-1]) if nums else None

rows, replies = [], {}
for name, build in prompts.items():
    results = run_all([build(e["question"]) for e in gsm_test])
    replies[name] = results
    correct = [extract_number(r["reply"]) == g for r, g in zip(results, gold_num)]
    rows.append({"Strategy": name, "Accuracy (%)": round(100 * sum(correct) / 30, 1),
                 "Avg tokens": round(sum(r["tokens"] for r in results) / 30),
                 "Avg seconds": round(sum(r["seconds"] for r in results) / 30, 2)})
gsm_table = pd.DataFrame(rows).set_index("Strategy")
gsm_table

,Accuracy (%),Avg tokens,Avg seconds
Strategy,,,
direct answer,36.7,73,0.89
zero-shot CoT,86.7,2549,5.96
few-shot CoT,90.0,464,1.70


### Step 5: Look at one problem under all three strategies

In [10]:
i = 0
print("QUESTION:", gsm_test[i]["question"], "| CORRECT:", gold_num[i])
for name in prompts:
    print(f"\n[{name}]\n" + replies[name][i]["reply"][:400])

QUESTION: Ashley's pizza delivery costs $15. What is the total amount that Ashley should give the delivery man if she wants to give a tip that is equal to 1/5 of the amount she ordered? | CORRECT: 18.0

[direct answer]
18

[zero-shot CoT]
Here's how to solve the problem step-by-step:

1.  **Identify the cost of the pizza:** Ashley's pizza delivery costs $15.

2.  **Determine the tip amount:** Ashley wants to give a tip that is equal to 1/5 of the amount she ordered.
    *   Calculate 1/5 of $15: (1/5) * $15 = $3.

3.  **Calculate the total amount:** The total amount is the cost of the pizza plus the tip.
    *   Total amount = Cost

[few-shot CoT]
Ashley ordered $15 worth of pizza.
She wants to give a tip that is equal to 1/5 of the amount she ordered.
So, the tip amount is (1/5) * $15 = $3.
The total amount Ashley should give the delivery man is the cost of the pizza plus the tip.
Total amount = $15 + $3 = $18.
Answer: 18


## 7. Experimental Observations

### Observation Table 1: Emotion classification (60 test sentences)

In [11]:
emotion_table

,Accuracy (%),Avg tokens
Strategy,,
zero-shot,61.7,57
few-shot (6 examples),66.7,235
few-shot (18 examples),66.7,595


### Observation Table 2: Maths word problems (30 test problems)

In [12]:
gsm_table

,Accuracy (%),Avg tokens,Avg seconds
Strategy,,,
direct answer,36.7,73,0.89
zero-shot CoT,86.7,2549,5.96
few-shot CoT,90.0,464,1.70


**Observations**

- Emotion classification: zero-shot reached 61.7 percent. Six examples raised it to 66.7 percent, and eighteen examples gave no further gain (66.7 percent) while the prompt grew from 57 to 595 tokens. The gain is modest because the labels of this dataset are noisy, and several sentences could reasonably belong to two emotions.
- Maths word problems: the direct answer was correct for only 36.7 percent of the problems. Zero-shot CoT raised accuracy to 86.7 percent and few-shot CoT to 90.0 percent, so asking for the reasoning more than doubled the accuracy.
- Zero-shot CoT was the most expensive strategy (about 2,500 tokens and 6 seconds per problem) because the model wrote long, formatted explanations. Few-shot CoT gave the best accuracy with only about 460 tokens and 1.7 seconds, because the worked examples showed the model a short answer style.
- For the sample problem the direct answer (18) was correct, but it is not reliable on harder problems; the CoT replies show each calculation, which also makes the answers easy to check.

## 8. Result

Zero-shot, few-shot and Chain-of-Thought prompts were designed and evaluated using the Gemini model through the OpenAI SDK. On 60 emotion-classification sentences, accuracy improved from 61.7 percent (zero-shot) to 66.7 percent (few-shot). On 30 GSM8K maths problems, accuracy improved from 36.7 percent (direct answer) to 86.7 percent (zero-shot CoT) and 90.0 percent (few-shot CoT). The experiment quantified the accuracy gained by each prompting technique and its cost in tokens and time.

## 9. Conclusion

Prompt design changes the behaviour of the same model without any training. Giving labelled examples (few-shot) helps most when the task format or label meaning is not obvious, while asking for step-by-step reasoning (Chain-of-Thought) helps on problems that need several calculation steps. Both techniques increase the number of tokens and the response time, so the extra cost must be weighed against the gain in accuracy for each task.